# Exploración de Datos de Calidad del Aire — Guatemala y LATAM

**Proyecto:** Air Quality LATAM  
**Autor:** Rodrigo, Ingeniero Ambiental, Universidad Anáhuac Cancún  
**Objetivo:** Exploración inicial de datos, limpieza, estadística descriptiva, mapas y correlaciones.

---
**Estructura del notebook:**
1. Imports y configuración
2. Carga de datos (CSV o datos sintéticos)
3. Limpieza con `clean.py`
4. Estadística descriptiva con `descriptive.py`
5. Mapa interactivo con `maps.py`
6. Análisis de correlaciones con `correlations.py`
7. Cálculo de AQI con `aqi_index.py`

## 1. Imports y configuración

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib

warnings.filterwarnings('ignore')
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

# Agregar raíz del proyecto al path
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    POLLUTANTS, REGIONS, WHO_GUIDELINES_2021,
    AQI_CATEGORIES, FIGURES_DIR, PROCESSED_DIR
)

print(f'Proyecto: {PROJECT_ROOT}')
print(f'Contaminantes configurados: {POLLUTANTS}')
print(f'Regiones de interés: {list(REGIONS.keys())}')

## 2. Carga de datos

Se intenta cargar desde `data/raw/`. Si no hay archivo, se generan datos sintéticos de Guatemala para demostración.

In [ ]:
from ingestion.load_stations import load_csv_stations, generate_synthetic_data

# Buscar CSV en raw/
csv_files = list((PROJECT_ROOT / 'data' / 'raw').glob('*.csv'))

if csv_files:
    print(f'Archivos CSV encontrados: {[f.name for f in csv_files]}')
    gdf = load_csv_stations(str(csv_files[0]))
    print(f'Datos cargados: {len(gdf)} filas')
else:
    print('Sin archivos CSV en data/raw/ — generando datos sintéticos de Guatemala...')
    df_sint = generate_synthetic_data(
        region='Guatemala',
        n_stations=8,
        days=180,
        output_path=str(PROJECT_ROOT / 'data' / 'raw' / 'guatemala_sintetico.csv')
    )
    df_sint['datetime'] = pd.to_datetime(df_sint['fecha'], format='%d/%m/%Y %H:%M')
    geometry = [Point(row['lon'], row['lat']) for _, row in df_sint.iterrows()]
    gdf = gpd.GeoDataFrame(df_sint, geometry=geometry, crs='EPSG:4326')
    print(f'Datos sintéticos generados: {len(gdf)} filas, {gdf["station_id"].nunique()} estaciones')

gdf.head()

In [ ]:
# Resumen del dataset
print('=== Información del dataset ===')
print(f'Filas: {len(gdf):,}')
print(f'Columnas: {list(gdf.columns)}')
print()

if 'datetime' in gdf.columns:
    print(f'Período: {gdf["datetime"].min()} → {gdf["datetime"].max()}')

if 'station_id' in gdf.columns:
    print(f'Estaciones: {gdf["station_id"].nunique()}')
    print(f'Estaciones: {sorted(gdf["station_id"].unique().tolist())}')

print()
print('Estadísticas básicas por contaminante:')
contaminantes_presentes = [p for p in POLLUTANTS if p in gdf.columns]
print(gdf[contaminantes_presentes].describe().round(2))

## 3. Limpieza con `clean.py`

In [ ]:
from processing.clean import AirQualityCleaner, clean_pipeline

# Inicializar limpiador
cleaner = AirQualityCleaner(gdf)

print(f'Contaminantes detectados: {cleaner.pollutant_cols}')
print(f'Datos antes de limpieza: {len(gdf):,} filas')

# Ejecutar pipeline completo
df_limpio = cleaner.clean_pipeline(
    outlier_method='iqr',
    missing_method='interpolate',
    max_gap_hours=6,
)

print(f'Datos después de limpieza: {len(df_limpio):,} filas')
print(f'Diferencia: {len(gdf) - len(df_limpio)} filas eliminadas/modificadas')

In [ ]:
# Reporte de calidad
reporte = cleaner.generate_qc_report(df_limpio)

print('=== Reporte de Calidad ===')
print(f"Total registros: {reporte['n_total']:,}")
print(f"Rango fechas: {reporte['rango_fechas']}")
print(f"Duración: {reporte['duracion_dias']} días")
print(f"Estaciones: {reporte['n_estaciones']}")
print(f"Gaps detectados: {reporte['n_gaps_total']}")

print('\n% Datos válidos por contaminante:')
for k, v in reporte['pct_datos_validos'].items():
    barra = '█' * int(v / 5) + '░' * (20 - int(v / 5))
    print(f'  {k:8s}: {barra} {v:.1f}%')

## 4. Estadística descriptiva con `descriptive.py`

In [ ]:
from analysis.descriptive import (
    full_descriptive, temporal_decomposition,
    diurnal_pattern, exceedance_stats, annual_trend
)

VARIABLE = 'PM2.5'  # Cambiar aquí para analizar otro contaminante

if VARIABLE not in df_limpio.columns:
    VARIABLE = contaminantes_presentes[0]
    print(f'PM2.5 no encontrado — usando: {VARIABLE}')

print(f'=== Estadísticas descriptivas de {VARIABLE} ===')
stats_df = full_descriptive(df_limpio, VARIABLE)
stats_df

In [ ]:
# Descomposición temporal
decomp = temporal_decomposition(df_limpio, VARIABLE)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle(f'Patrones temporales de {VARIABLE}', fontsize=13)

# Por hora
ax = axes[0, 0]
decomp['por_hora']['promedio'].plot(ax=ax, marker='o', color='steelblue', ms=4)
ax.set_title('Patrón diurno (por hora)')
ax.set_xlabel('Hora del día')
ax.set_ylabel(f'{VARIABLE} (µg/m³)')
ax.grid(True, alpha=0.3)

# Por día de semana
ax = axes[0, 1]
decomp['por_dia_semana']['promedio'].plot(ax=ax, kind='bar', color='coral', edgecolor='black')
ax.set_title('Patrón semanal')
ax.set_xlabel('')
ax.set_ylabel(f'{VARIABLE} (µg/m³)')
plt.setp(ax.get_xticklabels(), rotation=0)
ax.grid(True, alpha=0.3, axis='y')

# Por mes
ax = axes[1, 0]
decomp['por_mes']['promedio'].plot(ax=ax, kind='bar', color='mediumseagreen', edgecolor='black')
ax.set_title('Patrón mensual')
ax.set_xlabel('')
ax.set_ylabel(f'{VARIABLE} (µg/m³)')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(True, alpha=0.3, axis='y')

# Por año
ax = axes[1, 1]
if len(decomp['por_anio']) > 1:
    decomp['por_anio']['promedio'].plot(ax=ax, marker='s', color='purple', ms=6)
    ax.set_title('Tendencia anual')
    ax.set_ylabel(f'{VARIABLE} (µg/m³)')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Datos de un solo año\n(sin tendencia anual)',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Tendencia anual (insuficiente)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / f'patrones_temporales_{VARIABLE.replace(".","")}.png', dpi=130)
plt.show()
print(f'Gráfico guardado en {FIGURES_DIR}')

In [ ]:
# Estadísticas de excedencia OMS
exc = exceedance_stats(df_limpio, VARIABLE)

print(f'=== Excedencias de {VARIABLE} sobre umbral OMS ===')
print(f"Umbral utilizado: {exc['umbral']} µg/m³")
print(f"Excedencias: {exc['n_excedencias']}/{exc['n_total']} ({exc['pct_excedencias']:.1f}%)")
print(f"Concentración máxima: {exc['max_concentracion']} µg/m³")
if exc.get('mes_mayor_excedencia'):
    print(f"Mes con más excedencias: {exc['mes_mayor_excedencia']}")

## 5. Mapa rápido con `maps.py`

In [ ]:
from visualization.maps import interactive_map_folium

# Calcular promedio por estación
if 'station_id' in df_limpio.columns and 'lat' in df_limpio.columns:
    df_promedios = (
        df_limpio.groupby('station_id')
        .agg({VARIABLE: 'mean', 'lat': 'first', 'lon': 'first'})
        .reset_index()
        .dropna()
    )
    geometry_prom = [Point(row['lon'], row['lat']) for _, row in df_promedios.iterrows()]
    gdf_promedios = gpd.GeoDataFrame(df_promedios, geometry=geometry_prom, crs='EPSG:4326')
    
    output_html = str(PROJECT_ROOT / 'outputs' / 'maps' / f'mapa_{VARIABLE.replace(".","")}.html')
    path = interactive_map_folium(
        gdf_promedios,
        VARIABLE,
        output_path=output_html,
        station_col='station_id',
    )
    
    if path:
        print(f'Mapa guardado en: {path}')
        print('Para ver el mapa, abrir el archivo HTML en el navegador o usar el siguiente código:')
        print(f'from IPython.display import IFrame; IFrame("{path}", width=700, height=500)')
    else:
        print('Mapa no generado (verificar que folium esté instalado: pip install folium)')
else:
    print('No hay columnas de estación/coordenadas para el mapa')

In [ ]:
# Mapa estático rápido con matplotlib
if 'lat' in df_limpio.columns and VARIABLE in df_limpio.columns:
    df_prom_est = (
        df_limpio.groupby(['station_id', 'lat', 'lon'])[VARIABLE]
        .mean()
        .reset_index()
    )
    
    fig, ax = plt.subplots(figsize=(9, 7))
    sc = ax.scatter(
        df_prom_est['lon'], df_prom_est['lat'],
        c=df_prom_est[VARIABLE], cmap='YlOrRd',
        s=200, edgecolors='black', linewidths=0.8,
        zorder=5,
    )
    plt.colorbar(sc, ax=ax, label=f'{VARIABLE} promedio (µg/m³)')
    
    for _, row in df_prom_est.iterrows():
        ax.annotate(row['station_id'], (row['lon'], row['lat']),
                    fontsize=7, ha='center', va='bottom', xytext=(0, 8),
                    textcoords='offset points')
    
    ax.set_xlabel('Longitud')
    ax.set_ylabel('Latitud')
    ax.set_title(f'Concentración promedio de {VARIABLE} por estación')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'outputs' / 'maps' / f'conc_{VARIABLE.replace(".","")}.png', dpi=130)
    plt.show()

## 6. Correlaciones básicas con `correlations.py`

In [ ]:
from analysis.correlations import correlation_matrix, plot_correlation_heatmap

variables_analizar = [p for p in POLLUTANTS if p in df_limpio.columns]
print(f'Variables a correlacionar: {variables_analizar}')

if len(variables_analizar) >= 2:
    corr_result = correlation_matrix(df_limpio, variables_analizar, method='both')
    
    print('\n=== Correlación de Pearson ===')
    print(corr_result['pearson']['r'].round(3).to_string())
    
    print('\n=== P-values de Pearson (< 0.05 = significativo) ===')
    print(corr_result['pearson']['p'].round(3).to_string())
else:
    print('Se necesitan al menos 2 variables. Usando correlación de una variable con tiempo.')

In [ ]:
# Heatmap de correlaciones
if len(variables_analizar) >= 2:
    path_heatmap = plot_correlation_heatmap(
        corr_result['pearson']['r'],
        p_matrix=corr_result['pearson']['p'],
        title=f'Correlaciones entre contaminantes — {REGIONS["Guatemala"]["descripcion"]}',
    )
    print(f'Heatmap guardado en: {path_heatmap}')
    
    # Mostrar en notebook
    img = plt.imread(str(path_heatmap))
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(img)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

## 7. Cálculo de AQI con `aqi_index.py`

In [ ]:
from processing.aqi_index import aqi_dataframe, calculate_aqi, calculate_nowcast_pm25

# Calcular AQI para el dataset
df_aqi = aqi_dataframe(df_limpio, pollutant=VARIABLE, standard='EPA')

print(f'=== Distribución de categorías AQI para {VARIABLE} ===')
distribucion = df_aqi['categoria'].value_counts()
total = len(df_aqi['categoria'].dropna())
for cat, n in distribucion.items():
    barra = '█' * int(n / total * 30)
    print(f'  {cat:25s}: {barra} {n:6,} ({n/total*100:.1f}%)')

df_aqi[['datetime', VARIABLE, 'aqi', 'categoria', 'color']].head(10)

In [ ]:
# Gráfico de AQI en el tiempo
if 'datetime' in df_aqi.columns and 'aqi' in df_aqi.columns:
    df_aqi_sorted = df_aqi.sort_values('datetime')
    df_daily_aqi = (
        df_aqi_sorted
        .resample('D', on=pd.to_datetime(df_aqi_sorted['datetime']))
        ['aqi'].mean()
        .reset_index()
    )
    
    fig, ax = plt.subplots(figsize=(12, 4))
    
    # Franjas de color por categoría AQI
    bandas = [
        (0, 50, '#00E40025'),
        (51, 100, '#FFFF0025'),
        (101, 150, '#FF7E0025'),
        (151, 200, '#FF000025'),
        (201, 300, '#8F3F9725'),
    ]
    for lo, hi, color in bandas:
        ax.axhspan(lo, hi, alpha=0.15, color=color)
    
    ax.plot(df_daily_aqi['datetime'], df_daily_aqi['aqi'],
            color='#2E86AB', lw=1.5, alpha=0.8)
    ax.fill_between(df_daily_aqi['datetime'], df_daily_aqi['aqi'],
                   alpha=0.2, color='#2E86AB')
    
    ax.set_title(f'AQI diario — {VARIABLE}', fontsize=12)
    ax.set_xlabel('Fecha')
    ax.set_ylabel('AQI')
    ax.set_ylim(0, 300)
    ax.grid(True, alpha=0.3)
    
    # Etiquetas de categoría
    for nombre, info in {
        'Bueno': (25, '#00E400'), 'Moderado': (75, '#CCCC00'),
        'Insalubre\nsensibles': (125, '#FF7E00'), 'Insalubre': (175, '#FF0000'),
    }.items():
        aqi_val, color = info
        ax.text(df_daily_aqi['datetime'].iloc[-1], aqi_val, f' {nombre}',
               va='center', fontsize=7, color=color, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'aqi_{VARIABLE.replace(".","")}.png', dpi=130)
    plt.show()
    print(f'Gráfico AQI guardado en {FIGURES_DIR}')

In [ ]:
# NowCast para la última estación con datos
if 'station_id' in df_aqi.columns:
    ultima_estacion = df_aqi.iloc[-1]['station_id']
    df_est = df_aqi[df_aqi['station_id'] == ultima_estacion].sort_values('datetime')
    ultimas_12h = df_est[VARIABLE].tail(12).tolist()[::-1]  # más reciente primero
    
    nowcast = calculate_nowcast_pm25(ultimas_12h)
    if nowcast:
        aqi_nowcast = calculate_aqi(nowcast, 'PM2.5')
        print(f'=== NowCast PM2.5 — Estación {ultima_estacion} ===')
        print(f'Últimas 12h: {[round(v, 1) if v is not None else None for v in ultimas_12h]}')
        print(f'NowCast: {nowcast} µg/m³')
        print(f'AQI NowCast: {aqi_nowcast["aqi_value"]} — {aqi_nowcast["category"]}')
        print(f'Mensaje: {aqi_nowcast["health_message_es"]}')

---
## Resumen final

Este notebook ha demostrado el flujo completo de análisis:

1. **Carga de datos** — CSV con detección automática o datos sintéticos
2. **Limpieza** — duplicados, outliers, valores faltantes
3. **Estadística descriptiva** — métricas completas y patrones temporales
4. **Mapas** — visualización espacial de concentraciones
5. **Correlaciones** — relaciones entre contaminantes
6. **AQI** — cálculo EPA y NowCast

### Próximos pasos

- Descargar datos reales de OpenAQ: `python 01_ingestion/fetch_openaq.py`
- Lanzar el dashboard interactivo: `streamlit run 04_visualization/dashboard.py`
- Generar un reporte PDF: `python 04_visualization/reports.py`
- Analizar datos satelitales de polvo sahariano: `python 01_ingestion/fetch_satellite.py`